# 11-785/11-685 Recitation / Lab: Attention, Machine Translation, and LAS

**Please make a local copy of the notebook first!**

**Topics Covered:**  
- Attention Mechanism  
- Simple Neural Machine Translation (NMT) with GRUs  
- Connection to LAS (Listen, Attend, and Spell)

---

## Overview

In this lab, we will:

1. **Review** how a basic Sequence-to-Sequence (Seq2Seq) model works for machine translation.  
2. **Implement** a GRU-based encoder-decoder that includes an **attention mechanism**.  
3. **Train** the model on a small (toy) dataset to demonstrate the effect of attention.  
4. **Discuss** how this architecture ties into the LAS (Listen, Attend, and Spell) framework.

We will keep the dataset small and artificially constructed, so that training can happen quickly. The code and concepts, however, easily extend to larger, real-world machine translation tasks.

This notebook should take ~45-50 minutes to run through in-lab, with short “breakpoints” every 10-15 minutes to pause and discuss the concepts in depth.

---


In [ ]:
##########################
# 1. IMPORTS AND SETUP   #
##########################
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from typing import List, Tuple

# For reproducibility
SEED = 11785
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 2. Recap: Basic Seq2Seq (Without Attention)
A simple Seq2Seq model:
- **Encoder** processes the input sequence (e.g., ["I", "ate", "an", "apple", "<eos>"]) token by token.
- The final hidden state of the encoder is assumed to summarize the entire input sequence.
- **Decoder** initializes its hidden state using the encoder’s final hidden state. Then it generates the output sequence (e.g., ["<sos>", "Ich", "habe", "einen", "Apfel", "gegessen", "<eos>"]) token by token.

### The Problem
Placing all input information into just the *final* hidden state is often insufficient for longer or more complex sequences. Hence, **attention** was introduced to allow the decoder to look back at **all** encoder states, focusing on the most relevant parts at each decoding step.

---

## 3. Adding Attention

With attention:
- The encoder still produces a hidden state for each timestep (rather than just the final one).
- The decoder, at each output timestep, combines its own hidden state with *all* encoder hidden states (weighted appropriately) to produce a context vector.
- These weights (the “attention” or “alignment”) can shift from one part of the input sentence to another, as needed.

In the LAS framework (*Listen, Attend, and Spell*), the idea is similar:  
- **Listener** (encoder) - often a pyramidal or stacked LSTM/GRU that encodes the input (in speech tasks, often spectral frames).  
- **Attender** - computes a relevance score between the decoder’s state and all encoder states.  
- **Speller** (decoder) - uses the attention context + its own hidden state to generate the output (often one character or subword at a time).

We’ll implement a simplified version here using GRUs for both encoder and decoder, with a dot-product attention mechanism.

---


CHECKPOINT 1:

In [ ]:
###################################
# 4. DATASET (REVERSE COPY TASK)  #
###################################
# In the "reverse copy" task the model is given a random sequence of characters
# and is asked to output the reverse of that sequence.
#
# This is a well-known synthetic task to test sequence-to-sequence models,
# and with a sufficiently large training set and a small model, training can
# complete in just a few minutes.
#
# We'll generate a dataset of random sequences of characters using our vocabulary.

# Extend the vocabulary to include digits
SRC_VOCAB = ["<pad>", "<sos>", "<eos>"] + \
            [chr(i) for i in range(ord('a'), ord('z')+1)] + \
            [chr(i) for i in range(ord('0'), ord('9')+1)] + \
            [" "]
TGT_VOCAB = SRC_VOCAB  # Same vocab for both source and target
src2idx = {tkn: i for i, tkn in enumerate(SRC_VOCAB)}
idx2src = {i: tkn for i, tkn in enumerate(SRC_VOCAB)}
tgt2idx = {tkn: i for i, tkn in enumerate(TGT_VOCAB)}
idx2tgt = {i: tkn for i, tkn in enumerate(TGT_VOCAB)}

PAD_IDX = src2idx["<pad>"]
SOS_IDX =  # TODO
EOS_IDX =  # TODO

import string

def text_to_indices(text: str, mapping: dict):
    return [mapping[char] for char in text]

def indices_to_text(indices: list, mapping: dict):
    result = []
    for idx in indices:
        if mapping[idx] == "<eos>":
            break
        result.append(mapping[idx])
    return "".join(result)

def make_toy_pair(seq: str) -> tuple:
    """
    For the reverse copy task:
    Given an input sequence, the output is its reversal.
    Both sequences are tokenized into indices with <sos> and <eos>.
    """
    seq = seq.lower().strip()
    src_seq = [SOS_IDX] + text_to_indices(seq, src2idx) + [EOS_IDX]
    rev_seq = seq[::-1]
    tgt_seq = [SOS_IDX] + text_to_indices(rev_seq, tgt2idx) + [EOS_IDX]
    return src_seq, tgt_seq

import random

def generate_random_sequence(min_length=5, max_length=15):
    # Create a random sequence from lowercase letters, digits, and space.
    choices = string.ascii_lowercase + string.digits + " "
    length = random.randint(min_length, max_length)
    # Ensure we don't start or end with a space
    seq = ''.join(random.choices(choices, k=length)).strip()
    # If empty after strip, generate again
    return seq if seq != "" else generate_random_sequence(min_length, max_length)

# Generate a larger dataset for training:
num_train =  # TODO
train_data_pairs = [make_toy_pair(generate_random_sequence()) for _ in range(num_train)]

# For testing, generate a few examples
num_test =  # TODO
test_data_pairs = [make_toy_pair(generate_random_sequence()) for _ in range(num_test)]

# Pad sequences for batching
def pad_sequences(seqs: list) -> np.ndarray:
    max_len = max(len(seq) for seq in seqs)
    padded = []
    for seq in seqs:
        padded.append(seq + [PAD_IDX]*(max_len - len(seq)))
    return np.array(padded)

def generate_batch(data_pairs):
    random.shuffle(data_pairs)
    src_seqs = [pair[0] for pair in data_pairs]
    tgt_seqs = [pair[1] for pair in data_pairs]
    src_padded = pad_sequences(src_seqs)
    tgt_padded = pad_sequences(tgt_seqs)
    return torch.LongTensor(src_padded), torch.LongTensor(tgt_padded)

# Display a few examples (Don't change, this is here for testing)
print("Sample training data:")
for i, (src, tgt) in enumerate(train_data_pairs[:3]):
    print(f"Raw index src: {src}, tgt: {tgt}")
    print(f"Src str: {indices_to_text(src[1:-1], idx2src)}")
    print(f"Tgt str (should be reverse): {indices_to_text(tgt[1:-1], idx2tgt)}\n")


## 5. Model Architecture

### 5.1 Encoder (GRU)
- Embeds each source token into a continuous vector.
- Processes the embedded sequence with a GRU.
- Produces an output `hidden_state` for each time step (not just the final step).

### 5.2 Attention (Dot-Product)
- At each decoding step, we compute **attention scores** between the decoder's current hidden state and *all* encoder hidden states.
- We then take a weighted sum of the encoder states, using the normalized attention scores as weights.
  $$
  [
    \text{score}(h_{\text{dec}}, H_{\text{enc}}) = h_{\text{dec}} \cdot H_{\text{enc}}^\top
  ]
  $$
  $$
  [
    \alpha = \text{softmax}(\text{score})
  ]
  $$
  $$
  [
    \text{context} = \alpha \times H_{\text{enc}}
  ]
  $$

### 5.3 Decoder (GRU)
- At each decoding step, the decoder:
  1. Takes the *previous* target token (and/or embedding) plus the *context vector* from attention as input.
  2. Updates its hidden state via a GRU.
  3. Predicts the next token with a linear + softmax layer.

We’ll implement a single-layer GRU for both encoder and decoder for simplicity.


CHECKPOINT 2:

In [ ]:
###################################
# 6. ENCODER, ATTENTION, DECODER  #
###################################

class EncoderGRU(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, src):
        # src shape = (batch_size, src_len)
        embedded = self.embedding(src)
        # embedded shape = (batch_size, src_len, embed_dim)

        outputs, hidden = self.gru(embedded)
        # outputs shape = (batch_size, src_len, hidden_dim)
        # hidden shape = (num_layers, batch_size, hidden_dim)

        return outputs, hidden


class DotAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # For dot product attention, we typically don't need trainable parameters.
        # It's just a dot product + softmax.
        self.hidden_dim = hidden_dim

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (batch_size, hidden_dim) [the decoder’s current hidden state]
        encoder_outputs: (batch_size, src_len, hidden_dim)
        return: context vector (batch_size, hidden_dim),
                attention weights (batch_size, src_len)
        """
        # Expand decoder hidden for batched dot product
        # decoder_hidden shape: (batch_size, hidden_dim) -> (batch_size, 1, hidden_dim)
        dec_hidden_expanded = decoder_hidden.unsqueeze(1)

        # Dot product: (batch_size, 1, hidden_dim) * (batch_size, src_len, hidden_dim) -> (batch_size, 1, src_len)
        # .bmm is batch matrix multiplication
        attn_scores = torch.bmm(dec_hidden_expanded, encoder_outputs.transpose(1, 2))
        # attn_scores shape = (batch_size, 1, src_len)

        attn_weights =  # TODO
        # attn_weights shape = (batch_size, 1, src_len)

        # Weighted sum of encoder_outputs using attn_weights
        context =  # TODO
        # context shape = (batch_size, 1, hidden_dim)

        # Squeeze out the middle dimension
        context = context.squeeze(1)
        attn_weights = # TODO # (batch_size, src_len)

        return context, attn_weights


class DecoderGRU(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_dim, embed_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(embed_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

        # We will inject the attention module externally
        # because we want more flexibility

    def forward(self, tgt_token, prev_hidden, context_vector):
        """
        Single decoding step.
        tgt_token: (batch_size) current token index
        prev_hidden: (num_layers, batch_size, hidden_dim)
        context_vector: (batch_size, hidden_dim)
        """
        # Get embedding of current token
        embedded = self.embedding(tgt_token)  # shape (batch_size, embed_dim)
        # We need to combine embedded + context vector as input to GRU:
        # But GRU expects shape (batch_size, seq_len=1, input_size=?)
        combined_input = torch.cat([embedded, context_vector], dim=1).unsqueeze(1)

        output, hidden =  # TODO
        # output shape = (batch_size, 1, hidden_dim)
        # hidden shape = (num_layers, batch_size, hidden_dim)

        # Convert output to vocabulary distribution
        logits = self.fc_out(output.squeeze(1))  # shape = (batch_size, output_dim)

        return logits, hidden


## 7. Full Seq2Seq Model (Tying It All Together)

Now we'll combine the **EncoderGRU**, **DotAttention**, and **DecoderGRU** into a single module that:
1. Encodes the entire source sequence (generating encoder_outputs).
2. Iterates over the target sequence, at each step:
   - Applies attention to get context vector.
   - Feeds the previous token embedding + context into the decoder GRU.
   - Predicts the next token distribution.


CHECKPOINT 3:

In [ ]:
class Seq2SeqAttention(nn.Module):
    def __init__(self,
                 encoder: EncoderGRU,
                 decoder: DecoderGRU,
                 attention: DotAttention,
                 device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.attention = attention
        self.device = device

    def forward(self, src, tgt):
        """
        src: (batch_size, src_len)
        tgt: (batch_size, tgt_len) [we assume it includes <sos> at the start, <eos> at the end]

        We'll do "teacher forcing" here: at each step, feed the *gold* token as input (rather than the predicted token).
        Return the logits for each time step.
        """
        batch_size, tgt_len = tgt.shape
        # Encode
        encoder_outputs, encoder_hidden = self.encoder(src)

        # Initialize decoder hidden with encoder_hidden
        # For a GRU, let's just pass the final hidden state of the encoder
        decoder_hidden = encoder_hidden

        # We'll store logits for each time step
        # shape = (batch_size, tgt_len, vocab_size)
        outputs = torch.zeros(batch_size, tgt_len, self.decoder.fc_out.out_features).to(self.device)

        # We can start with a zero context (or use encoder_outputs mean)
        # For simplicity, let's do zeros
        context_vector = torch.zeros(batch_size, self.decoder.hidden_dim).to(self.device)

        # We iterate over each time step
        for t in range(tgt_len):
            # 1) Get the input token for this step (teacher forcing)
            input_token = # TODO  # shape (batch_size)

            # 2) Use attention
            # We'll use the top layer of the decoder's hidden state for attention
            # (decoder_hidden has shape (num_layers, batch_size, hidden_dim))
            decoder_top = decoder_hidden[-1]  # shape (batch_size, hidden_dim)
            context_vector, attn_weights =  # TODO

            # 3) Decoder forward
            logits, decoder_hidden = # TODO

            # 4) Store logits
            outputs[:, t, :] = logits

        return outputs


## 8. Training Loop

We'll do a simple training loop on our toy dataset.  
Steps:
1. **Forward** pass: get `outputs` of shape `(batch_size, tgt_len, vocab_size)`.
2. **Compute loss** vs. the ground truth `tgt`.
3. **Backprop** and **update** weights.

We’ll do teacher forcing at 100% to keep it simpler for demonstration.  


CHECKPOINT 4:

In [ ]:
def train_model(model, data_pairs, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(1, epochs+1):
        src_batch, tgt_batch = generate_batch(data_pairs)
        src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)

        optimizer.zero_grad()
        outputs = model(src_batch, tgt_batch[:, :-1])
        # We feed the target up to second-last token into the model
        # so the model predicts from 0..(tgt_len-2) -> 1..(tgt_len-1)
        # The final time step’s <eos> is not needed as input.

        # outputs shape = (batch_size, tgt_len-1, vocab_size)
        # we want to compare to tgt_batch[:, 1:] (the "shifted" gold)
        # Flatten the predictions & gold
        outputs_reshaped = outputs.reshape(-1, outputs.size(-1))  # (batch_size*(tgt_len-1), vocab_size)
        tgt_gold = tgt_batch[:, 1:].reshape(-1)  # (batch_size*(tgt_len-1))

        loss = criterion(outputs_reshaped, tgt_gold)
        loss.backward()
        optimizer.step()

        if epoch % 2 == 0:
            print(f"Epoch {epoch}/{epochs}, Loss = {loss.item():.4f}")

# Initialize the network
INPUT_DIM = len(SRC_VOCAB)
OUTPUT_DIM = len(TGT_VOCAB)
EMBED_DIM = 128
HIDDEN_DIM = 512

encoder =  # TODO
attention =  # TODO
decoder =  # TODO

seq2seq =  # TODO

optimizer =  # TODO
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

train_model(seq2seq, train_data_pairs, optimizer, criterion, epochs=50)


## 9. Inference / Testing

Let's write a function to *predict* a translation (or in our toy case, a reversed string).

We will:
1. Encode the `src`.
2. Start the decoder with `<sos>`.
3. Repeatedly:
   - Compute attention -> context
   - Decoder step -> predict next token
   - Stop when we predict `<eos>` or reach a max length


CHECKPOINT 5:

In [ ]:
def translate(model, seq_text: str, max_len=50):
    model.eval()
    with torch.no_grad():
        src_seq, _ = make_toy_pair(seq_text)
        src_tensor = torch.LongTensor([src_seq]).to(device)
        encoder_outputs, encoder_hidden = model.encoder(src_tensor)
        decoder_hidden = encoder_hidden
        context_vector = torch.zeros(1, model.decoder.hidden_dim).to(device)
        tgt_token = torch.LongTensor([SOS_IDX]).to(device)
        output_tokens = []
        for _ in range(max_len):
            decoder_top = decoder_hidden[-1]
            context_vector, _ =  # TODO
            logits, decoder_hidden =  # TODO
            next_token =  # TODO
            if next_token.item() == EOS_IDX:
                break
            output_tokens.append(next_token.item())
            tgt_token =  # TODO
        return indices_to_text(output_tokens, idx2tgt)

# Test on a few examples:
for i in range(5):
    # Use one of the test data pairs for demonstration:
    src_indices, tgt_indices = test_data_pairs[i]
    src_text = indices_to_text(src_indices[1:-1], idx2src)
    predicted = translate(seq2seq, src_text)
    print(f"Input: {src_text}")
    print(f"Expected Reverse: {indices_to_text(tgt_indices[1:-1], idx2tgt)}")
    print(f"Model Prediction: {predicted}\n")


## 10. Discussion and Connection to LAS

- **Attention** helps the decoder “focus” on relevant parts of the input at each output step, rather than relying solely on the encoder’s final state.
- **LAS** (Listen, Attend, and Spell) is a speech recognition system that similarly:
  - **Listens** (encodes audio frames into hidden representations, often with a pyramidal encoder for length reduction).
  - **Attends** (uses an attention mechanism, e.g., dot-product or location-based, to combine the decoder state and encoder outputs).
  - **Spells** (decodes one character at a time using an LSTM/GRU-based decoder).

### Key takeaways:
- The attention mechanism drastically improves the ability to handle long sequences.  
- The same principle (attend to relevant encoder positions for each output token) drives many modern architectures, including the Transformer (multi-head self-attention).  
- In LAS for speech, the idea is identical, just with more specialized encoders and often a character-level or subword-level decoder.
---
